In [1]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 104.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 104.0 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conv

In [1]:
!pip install pypdf

  Using cached pypdf-7.0.1-py3-none-any.whl.metadata (7.6 kB)
Using cached pypdf-7.0.1-py3-none-any.whl (416 kB)


In [2]:
import chromadb
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [3]:
uploaded = files.upload()

Saving Day_1_RAG_Semantic_Search_Assignment.pdf to Day_1_RAG_Semantic_Search_Assignment.pdf


In [4]:
pdf_name = list(uploaded.keys())[0]
print(pdf_name)

Day_1_RAG_Semantic_Search_Assignment.pdf


In [5]:
reader = PdfReader(pdf_name)
print(len(reader.pages))

1


In [6]:
full_text = ""
for page in reader.pages:
  text =page.extract_text()

  full_text += text

print("PDF extracted successfully")
print(len(full_text))

PDF extracted successfully
1657


In [7]:
# Chunking
def text_chunks(text,chunk_size,overlap):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]

    chunks.append(chunk)

    start += chunk_size - overlap

  return chunks

In [8]:
chunks_350 = text_chunks(full_text,350,50)
print(len(chunks_350))

6


In [9]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
embeddings = model.encode(chunks_350)

In [11]:
print("embeddings shape :",embeddings.shape)

embeddings shape : (6, 384)


In [13]:
client = chromadb.Client()

In [14]:
collection = client.create_collection(
    name= "pdf_embedding_collection",
    embedding_function = None
)

In [15]:
ids = [
    f"chunk_{i}" for i in range(len(chunks_350))
]

In [16]:
print(ids)

['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3', 'chunk_4', 'chunk_5']


In [17]:
metadata =[
    {
        "source": pdf_name,
        "chunk_number": i
    } for i in range(len(chunks_350))

]

In [18]:
collection.add(
    ids =ids,
    documents= chunks_350,
    embeddings = embeddings,
    metadatas = metadata
)

In [19]:
stored_data = collection.get()

In [20]:
print("Total stored recods :", len(stored_data["ids"]))

Total stored recods : 6


In [21]:
print("Document :", stored_data["documents"][3])

Document : eval
 Retrieve the three most relevant documents for a given query.
 Display the documents and their similarity scores in descending order.
5. Interactive Semantic Search
 Create a continuous search system that accepts multiple queries.
 For every query, display the top 3 matching documents and their similarity scores.
 Stop the program when t


In [22]:
print("Meta data :", stored_data["metadatas"][2])

Meta data : {'chunk_number': 2, 'source': 'Day_1_RAG_Semantic_Search_Assignment.pdf'}
